In [ ]:
# Read one archived HTML file and extract its article fields.
from parse_article import parse_article

article_path = 's3://databricks-news-articles/kontrast.at/1-mai-wehren-wir-uns-gegen-den-klassenkampf-von-oben.html'
table_name = '`news-analytics`.silver.articles'
file_row = spark.read.format('binaryFile').load(article_path).select('path', 'content').first()
article = parse_article(file_row.content.decode('utf-8', errors='replace'))
if article is None:
    raise ValueError(f'Not an article: {file_row.path}')
print('HEADLINE:', article['headline'])
print('CANONICAL URL:', article['canonical_url'])
print('SOURCE FILE:', file_row.path)

In [ ]:
# Build a typed Silver candidate and inspect the table before merging.
from datetime import datetime
from pyspark.sql import functions as F

record = dict(article)
record['published_at'] = datetime.fromisoformat(article['published_at'])
record['modified_at'] = datetime.fromisoformat(article['modified_at']) if article['modified_at'] else None
record['source_path'] = file_row.path
silver_before = spark.table(table_name)
candidate = spark.createDataFrame([record], schema=silver_before.schema)
display(candidate.select('canonical_url', 'headline', 'source_path'))
display(silver_before.agg(
    F.count('*').alias('articles'),
    F.countDistinct('canonical_url').alias('distinct_urls'),
))
display(silver_before.filter(F.col('canonical_url') == record['canonical_url'])
    .select('canonical_url', 'headline', 'source_path'))

In [ ]:
# Insert the candidate only when its canonical URL is absent from Silver.
candidate.createOrReplaceTempView('article_candidate')
merge_result = spark.sql(f'''
MERGE INTO {table_name} AS target
USING article_candidate AS source
ON target.canonical_url = source.canonical_url
WHEN NOT MATCHED THEN INSERT *
''')
print('MERGE RESULT:', merge_result.collect())

In [ ]:
# Check the Silver row counts and the article selected by canonical URL.
silver_after = spark.table(table_name)
display(silver_after.agg(
    F.count('*').alias('articles'),
    F.countDistinct('canonical_url').alias('distinct_urls'),
))
display(silver_after.filter(F.col('canonical_url') == record['canonical_url'])
    .select('canonical_url', 'headline', 'source_path'))